In [295]:
import pandas as pd
from pathlib import Path

folder = Path("../extract/arquivos_brutos/")

df_pts = pd.read_parquet(folder / 'df_partidas.parquet')
df_pts.head(4)

,0,1,2,3,4,5,6,7,8,9,10,11
0,,CAMPEONATO,DATA,TIER,STAGE,GAME TIME,TIME A,TIME B,FORMATO (BO:1/3/5),RESULTADO,PTS TIME A,PTS TIME B
1,,,,,,,,,,,,
2,,Campeonato Mineiro de Esportes Eletrônicos 2026,05/07/2026,TIER S,KNOCKOUT STAGE,,-,-,BO5,A,20,-20
3,,Campeonato Mineiro de Esportes Eletrônicos 2026,05/07/2026,TIER S,KNOCKOUT STAGE,,-,-,BO5,A,0,0


In [296]:
df_pts.columns

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], dtype='int64')

In [297]:
#refazendo colunas
df_pts.columns = df_pts.iloc[0]
df_pts = df_pts.iloc[2:].reset_index(drop=True)


In [298]:
#tratando colunas
df_pts.columns = [
  valor.lower().replace(" ", "_").replace("/", "-").replace("(", "").replace(")", "")
  for valor in df_pts.columns
]


In [299]:
#Removendo os "-" de colunas de times vazios:

colunas = ["time_a", "time_b"]
df_pts[colunas] = df_pts[colunas].replace("-", pd.NA)


In [300]:
#Removendo linhas que possuem os 2 times vazios
df_pts[colunas] = df_pts[colunas].apply(
  lambda coluna: coluna.str.strip()
).replace(["", "-"],pd.NA)

In [301]:
df_pts = df_pts.dropna(
    subset=["time_a", "time_b"],
    how="all"
)

In [302]:
#Inserindo coluna de id para partidas
df_pts.insert(loc=0, column='id_partida', value=0)


In [303]:
#Adicionando valor a coluna de id
df_pts["id_partida"] = (
 range(1, len(df_pts) + 1)
)

In [304]:
#Convertendo coluna de data para datetime
df_pts['data'] = pd.to_datetime(
  df_pts['data'].str.strip(),
  format= "mixed",
  dayfirst=True,
  errors="raise"
)

In [305]:
#Convertendo coluna de tempo de jogo para timedelta
df_pts["game_time"] = pd.to_timedelta(
    df_pts["game_time"].str.strip().replace("-", pd.NA),
    errors="raise"
)

In [306]:
#Renomeando coluna de formato de jogo
df_pts = df_pts.rename(
  columns= {"formato_bo:1-3-5" : "match_format"}
)

In [307]:
#Convertendo as colunas de pontuaçao para int
colunas =['pts_time_a', 'pts_time_b']

for coluna in colunas:
  df_pts[coluna] = pd.to_numeric(
    df_pts[coluna], errors="coerce"
  ).astype("Int64")

In [308]:
#Sequencia de true e false para os times que ganharam ou nao
venceu_a = df_pts['resultado'] == 'A'
venceu_b = df_pts['resultado'] == 'B'

In [309]:
#Criando a condicao para substituir as letras pelos times:
#Nas linhas em que A venceu, preencha resultado com o valor de time_a da mesma linha.
df_pts.loc[venceu_a, 'resultado'] = df_pts.loc[venceu_a, 'time_a']

#Nas linhas em que B venceu, preencha resultado com o valor de time_b da mesma linha.
df_pts.loc[venceu_b, 'resultado'] = df_pts.loc[venceu_b, 'time_b']

In [310]:
df_pts.head(4)

,id_partida,,campeonato,data,tier,stage,game_time,time_a,time_b,match_format,resultado,pts_time_a,pts_time_b
3,1,,Campeonato Mineiro de Esportes Eletrônicos 2026,2026-07-05,TIER S,KNOCKOUT STAGE,NaT,MIG,GHOST GAMING,BO3,MIG,15,-15
4,2,,Campeonato Mineiro de Esportes Eletrônicos 2026,2026-07-05,TIER S,KNOCKOUT STAGE,NaT,MIG,GHOST GAMING,BO3,MIG,0,0
5,3,,Campeonato Mineiro de Esportes Eletrônicos 2026,2026-07-05,TIER S,KNOCKOUT STAGE,NaT,FEAR CLAN,GEN MG,BO3,GEN MG,-15,15
6,4,,Campeonato Mineiro de Esportes Eletrônicos 2026,2026-07-05,TIER S,KNOCKOUT STAGE,NaT,FEAR CLAN,GEN MG,BO3,GEN MG,0,0


In [311]:
import re
import unicodedata

def normalizar_coluna(coluna):
  #Converter a coluna para string, remover espaços no inicio e no final, converter para minúsculas
  coluna = str(coluna).strip().lower()

  #Decomposição, retira acentos da palavra, ã vira a + ~
  coluna = unicodedata.normalize("NFKD", coluna)
  
  #Remove os acentos
  coluna = "".join(
    c for c in coluna
    if not unicodedata.combining(c)
  )

  coluna = re.sub(r"[^a-z0-9]+", "_", coluna)
  return coluna.strip("_")


In [312]:
#Ao inves de passar a coluna inteira para o metodo, aplicar o metodo a cada valor da coluna
df_pts['campeonato'] = df_pts['campeonato'].apply(normalizar_coluna)
df_pts['tier'] = df_pts['tier'].apply(normalizar_coluna)
df_pts['stage'] = df_pts['stage'].apply(normalizar_coluna)
df_pts['time_a'] = df_pts['time_a'].apply(normalizar_coluna)
df_pts['time_b'] = df_pts['time_b'].apply(normalizar_coluna)
df_pts['resultado'] = df_pts['resultado'].apply(normalizar_coluna)



In [313]:
equivalencias_campeonatos = {
  "sesc_geek": "sesc_geek_2024",
  
  "cmee_2025_conferencia_metropolitana":
        "campeonato_mineiro_de_esportes_eletronicos_2025_conferencia_metropolitana",

  "cmee_2025_conferencia_lcq":
    "campeonato_mineiro_de_esportes_eletronicos_2025_last_chance_qualify",

  "cmee_2025_conferencia_triangulo_mineiro":
    "campeonato_mineiro_de_esportes_eletronicos_2025_conferencia_triangulo",

  "liga_dinoco_s": "liga_dinoco_s_season_i",

  "desafio_uni_lol_circuito_academico_uni_bh":
    "uni_lol_circuito_academico_uni_bh",

  "arena_goat_lol": "arena_goat_lol_season_ii"
}

df_pts['campeonato'] = df_pts['campeonato'].replace(
  equivalencias_campeonatos
)

In [314]:
#Dimensao de torneios para fazer o cruzamento de dados
dim_torneios = pd.read_excel("excel_transform/campeonatos/Campeonatos.xlsx")


In [315]:
dim_torneios[['id_camp', 'nome_do_campeonato']].head()

,id_camp,nome_do_campeonato
0,0,campeonato_mineiro_de_esportes_eletronicos_2026
1,1,campeonato_mineiro_de_esportes_eletronicos_202...
2,2,campeonato_mineiro_de_esportes_eletronicos_202...
3,3,campeonato_mineiro_de_esportes_eletronicos_2025
4,4,campeonato_mineiro_de_esportes_eletronicos_202...


In [316]:
#Identificar valores repetidos e devolver uma sequencia de True e False
duplicados = dim_torneios['nome_do_campeonato'].duplicated(keep=False)

#Mostrar apenas as linhas marcadas como duplicadas, nas colunas que vou analisar
dim_torneios.loc[duplicados,['id_camp', 'nome_do_campeonato']]

,id_camp,nome_do_campeonato


In [317]:
#Realizando o cruzamento: Para cada partida, procurar o campeonato e trazer seu id:
partidas_com_torneio = df_pts.merge(
  dim_torneios[['id_camp', 'nome_do_campeonato']], #Seleciona ID e nome do campeonato
  left_on='campeonato',                            #Indica a coluna a esquerda a comparar
  right_on='nome_do_campeonato',                   #Indica a coluna a direita a comparar
  how='left',                                      #Mantem todas as partidas, se nao encontrar o camp, o id fica ausente
  validate='many_to_one',                          #Permite varias partidas para um mesmo torneio
  indicator=True                                   #Indica se houve correspondencia
)

In [318]:
partidas_com_torneio['_merge'].value_counts()

_merge
both          525
left_only       0
right_only      0
Name: count, dtype: int64

In [319]:
assert len(partidas_com_torneio) == len(df_pts), (
    "A quantidade de partidas mudou após o cruzamento."
)

assert partidas_com_torneio["id_camp"].notna().all(), (
    "Existem partidas sem ID de torneio."
)

In [320]:
#Transformando as informaçoes de campeonatos em IDs
df_pts = (
  partidas_com_torneio
  .rename(columns={"id_camp": "tournament_id"})
  .drop(columns=['campeonato', 'nome_do_campeonato', '_merge'])
  .copy()
)

df_pts['tournament_id'] = df_pts['tournament_id'].astype('int64')

In [321]:
df_pts[['id_partida', 'tournament_id', 'data']].head(6)

,id_partida,tournament_id,data
0,1,0,2026-07-05
1,2,0,2026-07-05
2,3,0,2026-07-05
3,4,0,2026-07-05
4,5,0,2026-07-04
5,6,0,2026-07-04
